# Bradford Bulls — Track-Level Annotation Pipeline (Colab)

Notebook chạy `v3/` pipeline trên Google Colab với **đúng workflow của host**: detect + track + filter (team color + ignore regions) → annotation package → optional Streamlit reviewer.

**Trước khi chạy:**
- Bật GPU: `Runtime` → `Change runtime type` → `T4 GPU` (hoặc tốt hơn).
- Mount Google Drive với video bạn muốn xử lý.

**Khác biệt so với phiên bản cũ:**
- Không force-downgrade `torch` / `numpy` của Colab → không còn dep conflict.
- Dùng workflow `--match-meta` (sidecar YAML) như host README → có team color filter + overlay mask, tránh tracking nhầm đối thủ / staff / scoreboard.
- Inline inspect: hiển thị số track + sample keyframe ngay trong notebook.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repository

Sửa `GIT_REPO` và `BRANCH` cho khớp repo của bạn.

In [ ]:
import os, subprocess

# TODO: chỉnh đường dẫn repo của bạn
GIT_REPO = 'https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git'
BRANCH   = 'main'              # branch chứa folder v3/
REPO_DIR = '/content/bradford_bulls'

# IMPORTANT: chdir về chỗ tồn tại TRƯỚC KHI rm -rf REPO_DIR.
# Nếu kernel đang đứng trong REPO_DIR (do %cd ở lần chạy trước) và ta rm -rf,
# shell sẽ mất CWD → git clone báo "Unable to read current working directory".
os.chdir('/content')
print(f'cwd = {os.getcwd()}')

if os.path.exists(REPO_DIR):
    print(f'Removing existing {REPO_DIR}')
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)

# Clone via subprocess so we get clear error if it fails (not nested in shell)
print(f'Cloning {GIT_REPO} (branch={BRANCH})...')
ret = subprocess.run(
    ['git', 'clone', '--depth', '1', '-b', BRANCH, GIT_REPO, REPO_DIR],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('GIT CLONE FAILED:')
    print(ret.stdout)
    print(ret.stderr)
    raise SystemExit(1)
print(ret.stdout or 'Clone OK')

V3_DIR = f'{REPO_DIR}/v3'
if not os.path.isdir(V3_DIR):
    print(f'\nv3/ not found at {V3_DIR}. Repo top-level contents:')
    for entry in sorted(os.listdir(REPO_DIR)):
        print(f'  {entry}')
    raise AssertionError(
        f'v3/ folder missing — check that branch {BRANCH!r} contains it, '
        f'or change BRANCH to one that does (e.g. feat/v3).'
    )

os.chdir(V3_DIR)
print(f'\ncwd = {os.getcwd()}')
subprocess.run(['ls', '-la'], check=True)

## 2b. Self-heal: đảm bảo `brands.yaml` tồn tại

Nếu repo bạn clone về thiếu `data/logo_templates/brands.yaml` (do `.gitignore` cũ vô tình loại nó), cell dưới sẽ tạo file từ phiên bản canonical — pipeline chạy được ngay.

Để fix vĩnh viễn: pull `.gitignore` mới (đã thêm `!data/logo_templates/brands.yaml`), commit + push file `brands.yaml`.

In [ ]:
import os
from pathlib import Path

brands_yaml_path = Path('data/logo_templates/brands.yaml')

# Canonical brands.yaml — keep in sync with v3/data/logo_templates/brands.yaml
BRANDS_YAML = """# ============================================================
# Brand registry — Bradford Bulls sponsorship logos
# ============================================================
#
# Structure:
#   brands[].id          — master brand identifier (used for reporting & metrics)
#   brands[].display_name — human-readable name (shown in reviewer UI)
#   brands[].variants[]  — appearance variants (light/dark/red/white/...)
#     .id                — unique variant identifier (used in YOLO multi-class export)
#     .kit_contexts      — list of kit contexts this variant appears in
#                          one or more of: home | away | special | any
#                          ("any" = always active regardless of kit)
#     .template_path     — path relative to data/logo_templates/
#
# Workflow:
#   - Each match has a kit_context (set via --kit-context CLI flag or match meta YAML)
#   - At annotation time, only variants whose kit_contexts include the match's
#     kit_context are shown to the annotator and considered for matching
#   - Brand-level reporting aggregates across all variants of the same brand
#
# Adding a new sponsor:
#   1. Add a new entry under `brands:`
#   2. Add 1+ variants (use kit_contexts: [any] if no kit-dependent variant exists)
#   3. Place template image(s) in the correct subfolder
#   4. Run: python scripts/normalize_logos.py to verify all paths exist
# ============================================================

brands:
  - id: aon
    display_name: "Aon"
    variants:
      - id: aon_red
        kit_contexts: [home]
        template_path: aon/red.png
      - id: aon_white
        kit_contexts: [away]
        template_path: aon/white.png

  - id: cch
    display_name: "CCH"
    variants:
      - id: cch_black
        kit_contexts: [home]
        template_path: cch/black.png
      - id: cch_white
        kit_contexts: [away]
        template_path: cch/white.png

  - id: mcp
    display_name: "MCP"
    variants:
      - id: mcp_home
        kit_contexts: [home]
        template_path: mcp/home.png
      - id: mcp_away
        kit_contexts: [away]
        template_path: mcp/away.png

  - id: paints_lacquers
    display_name: "Paints & Lacquers"
    variants:
      - id: paints_lacquers_red
        kit_contexts: [home]
        template_path: paints_lacquers/red.png
      - id: paints_lacquers_yellow
        kit_contexts: [away]
        template_path: paints_lacquers/yellow.png

  - id: romantica
    display_name: "Romantica Beds"
    variants:
      - id: romantica_white
        kit_contexts: [home]
        template_path: romantica/white.png
      - id: romantica_black
        kit_contexts: [away]
        template_path: romantica/black.png

  # ---- Single-variant brands (kit_contexts: [any]) ----

  - id: atm_hospitality
    display_name: "ATM Hospitality"
    variants:
      - id: atm_hospitality
        kit_contexts: [any]
        template_path: atm_hospitality.png

  - id: chadlaw
    display_name: "ChadLaw"
    variants:
      - id: chadlaw
        kit_contexts: [any]
        template_path: chadlaw.png

  - id: em_workwear
    display_name: "EM Workwear"
    variants:
      - id: em_workwear
        kit_contexts: [any]
        template_path: em_workwear.png

  - id: fairway_flooring
    display_name: "Fairway Flooring"
    variants:
      - id: fairway_flooring
        kit_contexts: [any]
        template_path: fairway_flooring.png

  - id: klg
    display_name: "KLG"
    variants:
      - id: klg
        kit_contexts: [any]
        template_path: klg.png

  - id: mna_cladding
    display_name: "MNA Cladding"
    variants:
      - id: mna_cladding
        kit_contexts: [any]
        template_path: mna_cladding.png

  - id: mna_support
    display_name: "MNA Support Services"
    variants:
      - id: mna_support
        kit_contexts: [any]
        template_path: mna_support.png

  - id: top_notch
    display_name: "Top Notch"
    variants:
      - id: top_notch
        kit_contexts: [any]
        template_path: top_notch.png

  - id: bartercard
    display_name: "Bartercard"
    variants:
      - id: bartercard
        kit_contexts: [any]
        template_path: bartercard.png

  - id: floor_tonic
    display_name: "Floor Tonic"
    variants:
      - id: floor_tonic
        kit_contexts: [any]
        template_path: floor_tonic.png

  - id: acs_group
    display_name: "ACS Group"
    variants:
      - id: acs_group
        kit_contexts: [any]
        template_path: acs_group.png
"""

if brands_yaml_path.exists():
    print(f'OK  brands.yaml already present ({brands_yaml_path.stat().st_size} bytes)')
else:
    brands_yaml_path.parent.mkdir(parents=True, exist_ok=True)
    brands_yaml_path.write_text(BRANDS_YAML)
    print(f'!!! brands.yaml was MISSING — wrote inline copy ({brands_yaml_path.stat().st_size} bytes)')
    print('    To fix permanently: commit data/logo_templates/brands.yaml to git')

# Quick parse check
import yaml
reg = yaml.safe_load(brands_yaml_path.read_text())
n_brands = len(reg["brands"])
n_variants = sum(len(b["variants"]) for b in reg["brands"])
print(f'    Loaded: {n_brands} brands, {n_variants} variants')

## 3. Cài dependencies — Colab-aware

Cell dưới **lọc requirements.txt cho Colab**: bỏ qua các package Colab đã có sẵn (`torch`, `numpy`, `opencv`, …) và bỏ qua các package optional có thể conflict (`mediapipe`). File `requirements.txt` không bị sửa — host vẫn dùng nguyên bản với caps đã test.

Chiến lược: **không đụng vào torch / torchvision / numpy / opencv của Colab**. Chỉ cài những thứ Colab thiếu (`ultralytics`, `pydantic-settings`, `loguru`, `streamlit`, `hydra-core`, …).

In [ ]:
# Filter requirements.txt for Colab compat:
#   COLAB_PRESERVE: package is pre-installed on Colab; don't reinstall (avoids force-downgrade)
#   COLAB_SKIP    : package is OPTIONAL on Colab (e.g., mediapipe pulls protobuf<5 conflict)
# Note: when requirements.txt has caps like torch<2.5.0 or numpy<2.0.0, pip would try to
# downgrade Colab's pre-installed torch/numpy. Skipping these here keeps Colab's stack intact.
COLAB_PRESERVE = {
    'torch', 'torchvision', 'torchaudio',
    'numpy', 'opencv-python', 'opencv-contrib-python',
    'pillow', 'matplotlib', 'pandas', 'scipy', 'scikit-learn',
    'jupyter', 'ipykernel', 'notebook', 'ipywidgets',
}
COLAB_SKIP = {
    'mediapipe',     # only needed by pose_align.py (lazy-imported); pulls protobuf<5 conflict
}

import re
src = open('requirements.txt').read().splitlines()
kept = []
skipped_preserve = []
skipped_optional = []
for line in src:
    s = line.strip()
    if not s or s.startswith('#'):
        continue
    pkg = re.split(r'[<>=!\s]', s)[0].lower()
    if pkg in COLAB_PRESERVE:
        skipped_preserve.append(s)
    elif pkg in COLAB_SKIP:
        skipped_optional.append(s)
    else:
        kept.append(s)

print('Skipped (use Colab pre-installed):')
for s in skipped_preserve: print('  -', s)
print('\nSkipped (optional / known conflict):')
for s in skipped_optional: print('  -', s)
print('\nWill install:')
for s in kept: print('  +', s)

with open('/tmp/requirements_colab.txt', 'w') as f:
    f.write('\n'.join(kept))

In [ ]:
# Install — quiet to giảm noise; --upgrade-strategy only-if-needed để không phá deps có sẵn
!pip install -q --upgrade-strategy only-if-needed -r /tmp/requirements_colab.txt

In [ ]:
# Verify import — KHÔNG cần restart runtime nếu chỉ cài các package trên (không động torch/numpy)
import torch, ultralytics, cv2, numpy, yaml
import pydantic, pydantic_settings, loguru, click
print(f'torch        {torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'ultralytics  {ultralytics.__version__}')
print(f'cv2          {cv2.__version__}')
print(f'numpy        {numpy.__version__}')
print(f'pydantic     {pydantic.VERSION}')
print(f'pydantic-settings ok')
print(f'loguru ok')
assert torch.cuda.is_available(), 'GPU not available — bật T4 GPU runtime trước!'

## 4. Tải YOLO weights

In [ ]:
import os
os.makedirs('weights', exist_ok=True)
if not os.path.exists('weights/yolo11l.pt'):
    !wget -q -O weights/yolo11l.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11l.pt
!ls -la weights/

## 5. Validate setup

Check brand registry, weights, deps. Lưu ý: nếu chưa chạy `scripts/normalize_logos.py` (chưa có 21 template image) sẽ có WARN — không sao, vì notebook này chỉ chạy detect+track, không cần template.

In [ ]:
!python scripts/validate_setup.py

## 6. Copy video từ Drive sang Colab

In [ ]:
import shutil
from pathlib import Path

# TODO: đường dẫn video trên Drive của bạn
VIDEO_DRIVE_PATH = '/content/drive/MyDrive/RugbyData/match_test.mp4'
VIDEO_NAME       = Path(VIDEO_DRIVE_PATH).stem

src = Path(VIDEO_DRIVE_PATH)
dst = Path(f'data/videos/{src.name}')
dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    shutil.copy2(src, dst)
print(f'Video local : {dst}  ({dst.stat().st_size / 1e6:.1f} MB)')

## 7. Tạo `match.meta.yaml` (recommended workflow)

Đây là phần **quan trọng nhất** để pipeline track đúng. Sidecar YAML này khai báo:
- `kit_context`: `home` / `away` / `special` → quyết định variant nào active
- `target_team.primary_colors`: HSV ranges cho jersey Bradford → filter bỏ đối thủ/staff
- `ignore_regions`: vùng overlay UI (scoreboard, channel logo) → bỏ qua khi detect

**Nếu skip cell này** và chỉ truyền `--kit-context`, pipeline sẽ track mọi người trong frame (cả đội đối thủ, trọng tài, khán giả) và cả vùng UI overlay → **package có hàng nghìn track sai**.

Cell dưới tạo meta cho **Bradford home (red+amber)** với template overlay BullsTV chuẩn. Sửa cho phù hợp nếu trận của bạn khác.

In [ ]:
META_YAML = f'''
kit_context: home
match_date: "2026-04-15"
opponent: "Hull FC"
venue: "Odsal"

target_team:
  primary_colors:
    - {{name: red,       h: [0, 10],    s: [120, 255], v: [70, 255]}}
    - {{name: red_wrap,  h: [170, 180], s: [120, 255], v: [70, 255]}}
    - {{name: amber,     h: [15, 30],   s: [100, 255], v: [120, 255]}}
  min_team_score: 0.10

ignore_regions:
  - [0.00, 0.85, 0.50, 1.00]    # bottom-left scoreboard
  - [0.85, 0.00, 1.00, 0.15]    # top-right BullsTV logo
'''

META_PATH = f'data/videos/{VIDEO_NAME}.meta.yaml'
Path(META_PATH).write_text(META_YAML.strip() + '\n')
print(f'Wrote {META_PATH}:')
print(Path(META_PATH).read_text())

## 8. Run pipeline (workflow chuẩn host)

Match host README workflow: `--match-meta` thay vì `--kit-context` thuần. `--max-duration 60` để test nhanh; bỏ flag này khi chạy thật.

In [ ]:
OUTPUT_DIR = f'data/annotation_packages/{VIDEO_NAME}'
MAX_DURATION = 60     # giây — None để xử lý cả trận

args = [
    'python', 'scripts/run_pipeline.py',
    '--video',      str(dst),
    '--config',     'configs/person_tracking.yaml',
    '--output',     OUTPUT_DIR,
    '--match-meta', META_PATH,
]
if MAX_DURATION:
    args += ['--max-duration', str(MAX_DURATION)]

print('Run:', ' '.join(args), '\n')

# Stream stdout/stderr line-by-line to the notebook so we see the real error
# instead of the opaque CalledProcessError. STDOUT and STDERR are merged so
# we see them in the order they were emitted.
import subprocess, sys
proc = subprocess.Popen(
    args,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='')
proc.wait()

if proc.returncode != 0:
    print(f'\n!!! Pipeline failed with exit code {proc.returncode}')
    print('Common causes:')
    print('  - weights/yolo11l.pt missing  → re-run section 4')
    print('  - data/logo_templates/brands.yaml missing  → check repo branch')
    print('  - meta YAML syntax error  → re-run section 7 and inspect output')
    print('  - GPU OOM  → lower detection.imgsz in configs/person_tracking.yaml (e.g. 960)')
    print('  - video codec unreadable  → verify with: !ffprobe data/videos/<file>')
    raise SystemExit(proc.returncode)

## 9. Inspect kết quả

Xem nhanh stats và một sample track để verify pipeline track đúng cầu thủ Bradford (không phải staff / scoreboard / đối thủ).

In [ ]:
!python -m track_annotation.cli inspect --package {OUTPUT_DIR}

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display, Markdown

manifest = json.loads((Path(OUTPUT_DIR) / 'manifest.json').read_text())
tracks = sorted((Path(OUTPUT_DIR) / 'tracks').iterdir())

display(Markdown(
    f"**Match**: {manifest['video']['filename']}  \n"
    f"**Resolution**: {manifest['video']['width']}×{manifest['video']['height']}  \n"
    f"**Kit context**: `{manifest['match_context']['kit_context']}`  \n"
    f"**Active brands**: {len(manifest['logo_templates'].get('active_brands', []))}  \n"
    f"**Tracks**: {manifest['stats']['num_tracks']}  \n"
    f"**Mean track duration**: {manifest['stats']['mean_track_duration_s']:.1f}s"
))

print(f'\nFirst 5 tracks:')
for td in tracks[:5]:
    meta = json.loads((td / 'meta.json').read_text())
    print(
        f"  {td.name}  frames={meta['num_frames']:3d}  dur={meta['duration_s']:5.1f}s  "
        f"area={meta['mean_area_ratio']*100:4.1f}%  conf={meta['mean_confidence']:.2f}  "
        f"team_score={meta.get('mean_team_score', 0):.2f}"
    )

In [ ]:
# Show keyframes của track đầu tiên
if tracks:
    td = tracks[0]
    print(f'Keyframes của {td.name}:')
    for img in sorted(td.glob('keyframe_*_full.jpg')):
        print(f'  {img.name}')
        display(Image(str(img), width=900))

## 10. (Optional) Launch Streamlit reviewer qua tunnel

Streamlit chạy local trên Colab nhưng cần tunnel để truy cập từ browser. Dùng `cloudflared` (miễn phí, không cần đăng ký).

**Annotate trên reviewer** → kết quả ghi vào `annotations.jsonl` trong package, dùng `--format yolo` ở cell sau để export training set.

In [ ]:
# Cài cloudflared (tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Launch Streamlit + tunnel ở background. Cell này không tự stop — xem URL log để mở.
import subprocess, threading, time

PORT = 8501
streamlit_cmd = [
    'streamlit', 'run', 'src/track_annotation/reviewer/app.py',
    '--server.port', str(PORT),
    '--server.headless', 'true',
    '--', '--package', OUTPUT_DIR,
]
import os
env = os.environ.copy()
env['PYTHONPATH'] = f'{V3_DIR}/src:' + env.get('PYTHONPATH', '')

streamlit_proc = subprocess.Popen(streamlit_cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(8)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Đang khởi động tunnel — sau ~5 giây xem URL `https://...trycloudflare.com` ở log dưới:\n')
def stream():
    for line in tunnel_proc.stdout:
        print(line.decode(errors='ignore'), end='')
threading.Thread(target=stream, daemon=True).start()
time.sleep(15)

In [ ]:
# Stop streamlit + tunnel khi annotate xong
try:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Stopped.')
except NameError:
    pass

## 11. Lưu kết quả ngược lên Drive

In [ ]:
ZIP_PATH = f'/content/{VIDEO_NAME}_package.zip'
!cd data/annotation_packages && zip -qr {ZIP_PATH} {VIDEO_NAME}

DRIVE_OUT = '/content/drive/MyDrive/RugbyData/'
!mkdir -p {DRIVE_OUT}
!cp {ZIP_PATH} {DRIVE_OUT}

import os
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Saved → {DRIVE_OUT}{VIDEO_NAME}_package.zip  ({size_mb:.1f} MB)')

## 12. (Optional) Export YOLO format sau khi annotate

Sau khi annotate xong qua Streamlit (cell #10), `annotations.jsonl` đã có data. Export sang YOLO format để train Stage A.

In [ ]:
# Brand-level multi-class (recommended cho training)
!python -m track_annotation.cli export \
    --package {OUTPUT_DIR} \
    --format yolo \
    --output data/yolo_dataset_{VIDEO_NAME} \
    --class-mode brand

In [ ]:
# Single-class (cho Stage A class-agnostic detector)
!python -m track_annotation.cli export \
    --package {OUTPUT_DIR} \
    --format yolo \
    --output data/yolo_dataset_{VIDEO_NAME}_stage_a \
    --class-mode single